In [5]:

import fasttext

# 1. Load the model once
print("Loading model...")
model = fasttext.load_model(r"C:\Users\NITRO V 15\Desktop\QUERYYY\model\QueryClassifier_6Classes.bin")

def build_pipeline_for_query(query: str) -> dict:
    """
    Takes a single query string, runs it through the classifier,
    and returns the exact RAG pipeline configurations.
    """
    cleaned = query.replace("\n", " ").lower().strip()
    
    # Fast path for short queries (under 5 words)
    if len(cleaned.split()) < 5:
        return {
            "query_type": "retrieval (fast-path)",
            "confidence": 1.0,
            "pipeline": {"retrievers": ["dense", "bm25"], "reranker": False, "expansion": None}
        }
        
    # Run FastText prediction
    labels, probabilities = model.predict(cleaned, k=1)
    predicted_label = labels[0].replace("__label__", "")
    confidence = probabilities[0]
    
    # Fallback to standard retrieval if model is unsure (< 60% confidence)
    if confidence < 0.60:
        predicted_label = "retrieval"
        
    # Map predictions to dynamic RAG pipeline settings
    pipeline_configs = {
        "retrieval":        {"retrievers": ["dense", "bm25"], "reranker": False, "expansion": None},
        "comparison":       {"retrievers": ["dense", "bm25"], "reranker": True, "expansion": None},
        "multi_hop":        {"retrievers": ["dense", "bm25", "graphrag"], "reranker": True, "expansion": "multi_query"},
        "summarization":    {"retrievers": ["dense"], "raptor_summaries": True, "reranker": False, "expansion": None},
        "metadata_filter":  {"retrievers": ["dense_filtered"], "self_query_filter": True, "reranker": False, "expansion": None},
        "follow_up":        {"retrievers": ["dense"], "history_memory": True, "reranker": False, "expansion": None}
    }
    
    return {
        "query_type": predicted_label,
        "confidence": float(confidence),
        "pipeline": pipeline_configs.get(predicted_label, pipeline_configs["retrieval"])
    }

# Interactive test
if __name__ == "__main__":
    # Get query from user
    query = input("\nEnter your query: ")
    
    # Build the pipeline configuration
    config = build_pipeline_for_query(query)
    
    print("\n--- Generated Pipeline Configuration ---")
    print(f"Query:      \"{query}\"")
    print(f"Route:      {config['query_type'].upper()}")
    print(f"Confidence: {config['confidence']:.2%}")
    print(f"Settings:   {config['pipeline']}")

Loading model...



--- Generated Pipeline Configuration ---
Query:      "which is best grocery shop in lucknow"
Route:      COMPARISON
Confidence: 99.98%
Settings:   {'retrievers': ['dense', 'bm25'], 'reranker': True, 'expansion': None}
